# Demo: Unterrichtsblock 4

Kurze Transferdemo für fehlende Werte, Duplikate und Zeitindex.

Du arbeitest hier mit synthetischen Daten von zwei Lüftungsanlagen (RLT) in einem Schulgebäude. Anschliessend bearbeitest du selbstständig `01-duplikate-fehlende_werte.ipynb` und `02-timeindex.ipynb`.

Relevante Artikel in der Wissensdatenbank:

- [Wissensdatenbank: Entfernen von Duplikaten](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/drop_duplicates/)
- [Wissensdatenbank: Befüllen von fehlenden Werten](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/fillna/)
- [Wissensdatenbank: Time-Indexes](https://abbts-dat-ski.github.io/wissensdatenbank/data-engineering/time_indexes/)

In [1]:
import pandas as pd

raw = pd.read_csv("../data/demo_rlt_school_building_raw.csv")
raw.head()


,MEASURED_AT_LOCAL,AHU_ID,ZONE,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
0,15.01.2026 07:30,RLT_NORD,KLASSENTRAKT_NORD,62,19.8,21.6,860.0,15.0,1.0
1,15.01.2026 07:00,RLT_SUED,LABORTRAKT_SUED,35,18.7,20.8,520.0,NaN,0.0
2,15.01.2026 07:45,RLT_NORD,KLASSENTRAKT_NORD,68,20.1,21.9,NaN,NaN,0.0
3,15.01.2026 07:15,RLT_NORD,KLASSENTRAKT_NORD,48,19.1,21.1,690.0,NaN,0.0
4,15.01.2026 07:00,RLT_NORD,KLASSENTRAKT_NORD,42,18.9,20.7,610.0,0.0,0.0


### Aufgabe 1
Bereinige fehlende Werte und Duplikate. Begründe für jede Spalte, welcher Füllwert fachlich sinnvoll ist.

- Kontext: zwei RLT-Anlagen im Schulhaus
- Überblick: wie viele Werte fehlen, Wertebereich pro Spalte
- fehlende Werte bei `MANUAL_OVERRIDE_MIN`, `ALARM_COUNT` und `EXTRACT_CO2_PPM`
- kein Eintrag bei Eingriff oder Alarm heisst: nichts passiert
- CO2-Sensorausfall: 0 ppm ist unmöglich
- Duplikate über `MEASURED_AT_LOCAL` + `AHU_ID`

In [2]:
# Dein Code hie
# Fehlende Werte pro Spalte
print(raw.isna().sum())

# Wertebereich pro Spalte
raw.describe()

MEASURED_AT_LOCAL      0
AHU_ID                 0
ZONE                   0
FAN_SPEED_PCT          0
SUPPLY_AIR_TEMP_C      0
RETURN_AIR_TEMP_C      0
EXTRACT_CO2_PPM        1
MANUAL_OVERRIDE_MIN    6
ALARM_COUNT            1
dtype: int64


,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
count,12.000000,12.000000,12.000000,11.000000,6.000000,11.000000
mean,55.583333,19.491667,21.408333,746.363636,6.666667,0.181818
std,11.649880,0.548483,0.483281,132.760892,7.527727,0.404520
min,35.000000,18.700000,20.700000,520.000000,0.000000,0.000000
25%,46.000000,19.000000,21.000000,640.000000,0.000000,0.000000
50%,60.000000,19.550000,21.500000,780.000000,5.000000,0.000000
75%,63.750000,19.850000,21.750000,855.000000,13.750000,0.000000
max,71.000000,20.400000,22.200000,940.000000,15.000000,1.000000


In [3]:
# Kein Eintrag = kein Eingriff / kein Alarm → 0 ist fachlich korrekt
raw["MANUAL_OVERRIDE_MIN"] = raw["MANUAL_OVERRIDE_MIN"].fillna(0)
raw["ALARM_COUNT"] = raw["ALARM_COUNT"].fillna(0)

In [4]:
import numpy as np

# 0 ppm ist physikalisch unmöglich (Aussenluft hat ca. 400 ppm) → als fehlend markieren
raw["EXTRACT_CO2_PPM"] = raw["EXTRACT_CO2_PPM"].replace(0, np.nan)

# Pro Anlage zeitlich sortiert interpolieren
raw = raw.sort_values(["AHU_ID", "MEASURED_AT_LOCAL"])
raw["EXTRACT_CO2_PPM"] = (
    raw.groupby("AHU_ID")["EXTRACT_CO2_PPM"]
      .transform(lambda s: s.interpolate().ffill().bfill())
)

In [5]:
print("Duplikate:", raw.duplicated(subset=["MEASURED_AT_LOCAL", "AHU_ID"]).sum())

raw = raw.drop_duplicates(subset=["MEASURED_AT_LOCAL", "AHU_ID"], keep="first")

Duplikate: 2


In [6]:
print(raw.isna().sum())
print("Verbleibende Duplikate:", raw.duplicated(subset=["MEASURED_AT_LOCAL", "AHU_ID"]).sum())

MEASURED_AT_LOCAL      0
AHU_ID                 0
ZONE                   0
FAN_SPEED_PCT          0
SUPPLY_AIR_TEMP_C      0
RETURN_AIR_TEMP_C      0
EXTRACT_CO2_PPM        0
MANUAL_OVERRIDE_MIN    0
ALARM_COUNT            0
dtype: int64
Verbleibende Duplikate: 0


### Aufgabe 2
Baue aus den bereinigten Daten eine Zeitreihe und zeige nur die Messungen zwischen 07:00 und 07:30.

- `MEASURED_AT_LOCAL` ist zuerst nur Text
- echte Zeitinformation
- als Index setzen und sortieren
- zeitbasierte Slices

In [7]:
raw["MEASURED_AT_LOCAL"] = pd.to_datetime(raw["MEASURED_AT_LOCAL"],dayfirst=True)
raw = raw.set_index("MEASURED_AT_LOCAL")
raw

,AHU_ID,ZONE,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
MEASURED_AT_LOCAL,,,,,,,,
2026-01-15 07:00:00,RLT_NORD,KLASSENTRAKT_NORD,42,18.9,20.7,610.0,0.0,0.0
2026-01-15 07:15:00,RLT_NORD,KLASSENTRAKT_NORD,48,19.1,21.1,690.0,0.0,0.0
2026-01-15 07:30:00,RLT_NORD,KLASSENTRAKT_NORD,62,19.8,21.6,860.0,15.0,1.0
2026-01-15 07:45:00,RLT_NORD,KLASSENTRAKT_NORD,68,20.1,21.9,900.0,0.0,0.0
2026-01-15 08:00:00,RLT_NORD,KLASSENTRAKT_NORD,71,20.4,22.2,940.0,0.0,0.0
2026-01-15 07:00:00,RLT_SUED,LABORTRAKT_SUED,35,18.7,20.8,520.0,0.0,0.0
2026-01-15 07:15:00,RLT_SUED,LABORTRAKT_SUED,46,19.0,21.0,640.0,0.0,0.0
2026-01-15 07:30:00,RLT_SUED,LABORTRAKT_SUED,58,19.4,21.4,780.0,0.0,0.0
2026-01-15 07:45:00,RLT_SUED,LABORTRAKT_SUED,63,19.7,21.7,820.0,10.0,0.0


In [8]:
raw.index

DatetimeIndex(['2026-01-15 07:00:00', '2026-01-15 07:15:00',
               '2026-01-15 07:30:00', '2026-01-15 07:45:00',
               '2026-01-15 08:00:00', '2026-01-15 07:00:00',
               '2026-01-15 07:15:00', '2026-01-15 07:30:00',
               '2026-01-15 07:45:00', '2026-01-15 08:00:00'],
              dtype='datetime64[us]', name='MEASURED_AT_LOCAL', freq=None)

In [10]:
raw=raw.sort_index()
raw

,AHU_ID,ZONE,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
MEASURED_AT_LOCAL,,,,,,,,
2026-01-15 07:00:00,RLT_NORD,KLASSENTRAKT_NORD,42,18.9,20.7,610.0,0.0,0.0
2026-01-15 07:00:00,RLT_SUED,LABORTRAKT_SUED,35,18.7,20.8,520.0,0.0,0.0
2026-01-15 07:15:00,RLT_NORD,KLASSENTRAKT_NORD,48,19.1,21.1,690.0,0.0,0.0
2026-01-15 07:15:00,RLT_SUED,LABORTRAKT_SUED,46,19.0,21.0,640.0,0.0,0.0
2026-01-15 07:30:00,RLT_NORD,KLASSENTRAKT_NORD,62,19.8,21.6,860.0,15.0,1.0
2026-01-15 07:30:00,RLT_SUED,LABORTRAKT_SUED,58,19.4,21.4,780.0,0.0,0.0
2026-01-15 07:45:00,RLT_NORD,KLASSENTRAKT_NORD,68,20.1,21.9,900.0,0.0,0.0
2026-01-15 07:45:00,RLT_SUED,LABORTRAKT_SUED,63,19.7,21.7,820.0,10.0,0.0
2026-01-15 08:00:00,RLT_NORD,KLASSENTRAKT_NORD,71,20.4,22.2,940.0,0.0,0.0


In [11]:
raw["2026-01-15 07:00":"2026-01-15 07:30"]

,AHU_ID,ZONE,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
MEASURED_AT_LOCAL,,,,,,,,
2026-01-15 07:00:00,RLT_NORD,KLASSENTRAKT_NORD,42,18.9,20.7,610.0,0.0,0.0
2026-01-15 07:00:00,RLT_SUED,LABORTRAKT_SUED,35,18.7,20.8,520.0,0.0,0.0
2026-01-15 07:15:00,RLT_NORD,KLASSENTRAKT_NORD,48,19.1,21.1,690.0,0.0,0.0
2026-01-15 07:15:00,RLT_SUED,LABORTRAKT_SUED,46,19.0,21.0,640.0,0.0,0.0
2026-01-15 07:30:00,RLT_NORD,KLASSENTRAKT_NORD,62,19.8,21.6,860.0,15.0,1.0
2026-01-15 07:30:00,RLT_SUED,LABORTRAKT_SUED,58,19.4,21.4,780.0,0.0,0.0


### Aufgabe 3
Interpretiere die Zeitstempel als Schweizer Ortszeit und rechne sie in UTC um. Was passiert bei der Umstellung auf Sommerzeit?

- naive Zeit kennt keine Zeitzone
- `Europe/Zurich` zuweisen
- in UTC umrechnen
- Frühling: 02:00 bis 02:59 fehlt

In [12]:
# Dein Code hier
raw = raw.tz_localize("Europe/Zurich")

In [13]:
raw = raw.tz_convert("UTC")

In [14]:
raw

,AHU_ID,ZONE,FAN_SPEED_PCT,SUPPLY_AIR_TEMP_C,RETURN_AIR_TEMP_C,EXTRACT_CO2_PPM,MANUAL_OVERRIDE_MIN,ALARM_COUNT
MEASURED_AT_LOCAL,,,,,,,,
2026-01-15 06:00:00+00:00,RLT_NORD,KLASSENTRAKT_NORD,42,18.9,20.7,610.0,0.0,0.0
2026-01-15 06:00:00+00:00,RLT_SUED,LABORTRAKT_SUED,35,18.7,20.8,520.0,0.0,0.0
2026-01-15 06:15:00+00:00,RLT_NORD,KLASSENTRAKT_NORD,48,19.1,21.1,690.0,0.0,0.0
2026-01-15 06:15:00+00:00,RLT_SUED,LABORTRAKT_SUED,46,19.0,21.0,640.0,0.0,0.0
2026-01-15 06:30:00+00:00,RLT_NORD,KLASSENTRAKT_NORD,62,19.8,21.6,860.0,15.0,1.0
2026-01-15 06:30:00+00:00,RLT_SUED,LABORTRAKT_SUED,58,19.4,21.4,780.0,0.0,0.0
2026-01-15 06:45:00+00:00,RLT_NORD,KLASSENTRAKT_NORD,68,20.1,21.9,900.0,0.0,0.0
2026-01-15 06:45:00+00:00,RLT_SUED,LABORTRAKT_SUED,63,19.7,21.7,820.0,10.0,0.0
2026-01-15 07:00:00+00:00,RLT_NORD,KLASSENTRAKT_NORD,71,20.4,22.2,940.0,0.0,0.0
